In [6]:
#This script reads in a list of CIFs with ASE and writes them back out again. 
#This seems a bit silly, but it's because the default formatting of the CIFs obtained from the CSD with C
#onQuest are not immediately suitable for use with a variety of Python packages like Pymatgen. 
#I recommend running this script first.


from ase.io import read, write
import os

cif_path = r'/Users/shubhamjamdade/Desktop/MOF_ready_for_water_simulations/ALL_DEFECTS/MOFs/'
cifs = os.listdir(cif_path)
cifs.sort()

for cif in cifs:
    
    mof = read(os.path.join(cif_path, cif))

    write(os.path.join(cif_path, cif), mof)

In [7]:
#This script converts a list of CIFs to their Niggli-reduced primitive cells. I recommend running this second.

from pymatgen.core import Structure
import os

folder = r'/Users/shubhamjamdade/Desktop/MOF_ready_for_water_simulations/ALL_DEFECTS/MOFs/'
for entry in os.listdir(folder):
    structure = Structure.from_file(os.path.join(folder,entry),primitive=True)
    structure.to(filename=os.path.join(folder,entry))

/Users/shubhamjamdade/opt/anaconda3/lib/python3.9/site-packages/pymatgen/core/__init__.py:49: UserWarning: Error loading .pmgrc.yaml: [Errno 2] No such file or directory: '/Users/shubhamjamdade/.pmgrc.yaml'. You may need to reconfigure your yaml file.
  warnings.warn(f"Error loading .pmgrc.yaml: {ex}. You may need to reconfigure your yaml file.")
/Users/shubhamjamdade/opt/anaconda3/lib/python3.9/site-packages/pymatgen/io/cif.py:1160: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: %s" % "\n".join(self.warnings))


In [9]:
#This script will check for small interatomic distances.

from ase.io import read
import os
import numpy as np

cutoff = 0.75  # interatomic distance threshold
folder = r'/Users/shubhamjamdade/Desktop/MOF_ready_for_water_simulations/ALL_DEFECTS/MOFs/'
bad_list = []
for cif in os.listdir(folder):
    mof = read(os.path.join(folder, cif))
    d = mof.get_all_distances()
    upper_diag = d[np.triu_indices_from(d, k=1)]
    for entry in upper_diag:

        if entry < cutoff:

            print('Interatomic distance issue:' + cif.split('.')[0])

            bad_list.append(cif)

            break

with open('bad_cifs_distance_check.txt','w') as w:

    for bad_cif in bad_list:

        w.write(bad_cif+'\n')

In [10]:
#This script will check for lone atoms in the framework, as determined using Pymatgen's CrystalNN tool.

from pymatgen.analysis.graphs import StructureGraph
from pymatgen.analysis import local_env
from pymatgen.core import Structure
import os

folder = r'/Users/shubhamjamdade/Desktop/MOF_ready_for_water_simulations/ALL_DEFECTS/MOFs/'
cifs = os.listdir(folder)
cifs.sort()
bad_list = []
for cif in cifs:
    mof = Structure.from_file(os.path.join(folder, cif))
    nn = local_env.CrystalNN()
    graph = StructureGraph.with_local_env_strategy(mof, nn)
    for j in range(len(mof)):
        nbr = graph.get_connected_sites(j)
    if not nbr:
            print('Lone atom issue:' + cif+'\n')
            bad_list.append(cif)
            break

with open('bad_cifs_lone_atom_check.txt','w') as w:
    for bad_cif in bad_list:
        w.write(bad_cif+'\n')

/Users/shubhamjamdade/opt/anaconda3/lib/python3.9/site-packages/pymatgen/analysis/local_env.py:4131: UserWarning: No oxidation states specified on sites! For better results, set the site oxidation states in the structure.
  warnings.warn(
/Users/shubhamjamdade/opt/anaconda3/lib/python3.9/site-packages/pymatgen/analysis/local_env.py:3934: UserWarning: CrystalNN: cannot locate an appropriate radius, covalent or atomic radii will be used, this can lead to non-optimal results.
  warnings.warn(


KeyboardInterrupt: 

In [11]:
#This script will check for missing H atoms on "terminal" metal-oxo species that should be terminal OH groups or terminal H2O groups.


from ase import neighborlist
from ase.io import read
import os
import numpy as np
import warnings

# Metals that should not have terminal oxo ligands
metals = ['Li','Na','K','Rb','Cs','Fr',
    'Be','Mg','Ca','Sr','Ba','Ra',
    'Sc','Y','La','Ac',
    'Ti','Zr','Hf',
    'Mn',
    'Fe',
    'Co',
    'Ni',
    'Cu','Ag',
    'Zn','Cd',
    'Al','Ga','In','Tl']

# Path to CIFs
p = r'/Users/shubhamjamdade/Desktop/MOF_ready_for_water_simulations/ALL_DEFECTS/MOFs/'

# Get CIFs from folder
cifs = os.listdir(p)
cifs = [cif for cif in cifs if '.cif' in cif]
cifs.sort()

# Check every CIF
bad_list = []
for cif in cifs:

    bad = False

    # Read in CIF, ignoring ASE warnings
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        structure = read(os.path.join(p,cif))

    # Get list of atomic symbols
    syms = np.array(structure.get_chemical_symbols())

    # Is one of the specified metals in this MOF
    if not any(item in syms for item in metals):
        continue

    # Initialize neighbor list
    cutoff = neighborlist.natural_cutoffs(structure)
    nl = neighborlist.NeighborList(cutoff,self_interaction=False,bothways=True)
    nl.update(structure)

    # For every site, check if it is a terminal metal-oxo
    for i, sym in enumerate(syms):

        # Confirm site is in pre-specified metal list
        if sym not in metals:
            continue

        # Get neighbors to metal
        bonded_atom_indices = nl.get_neighbors(i)[0]
        if bonded_atom_indices is None:
            continue
        bonded_atom_symbols = syms[bonded_atom_indices]

        # For every neighbor, check if it's a terminal oxo
        for j, bonded_atom_symbol in enumerate(bonded_atom_symbols):

            # Confirm neighbor is an O atom
            if bonded_atom_symbol != 'O':
                continue

            # Check if the O atom is only bound to the metal
            cn = len(nl.get_neighbors(bonded_atom_indices[j])[0])
            if cn == 1:
                bad = True
                print('Missing H on terminal oxo: ' + cif)
                bad_list.append(cif)

            if bad:
                break
    if bad:
            break

with open('bad_cifs_oxo_check.txt','w') as w:
    for bad_cif in bad_list:
        w.write(bad_cif + '\n')



In [12]:
#This script will de-duplicate a list of CIFs by using Pymatgen's StructureMatcher utility.

from pymatgen.core import Structure
from pymatgen.analysis import structure_matcher
import os


folder = r'/Users/shubhamjamdade/Desktop/MOF_ready_for_water_simulations/ALL_DEFECTS/MOFs/'
new_folder = r'/Users/shubhamjamdade/Desktop/MOF_ready_for_water_simulations/ALL_DEFECTS/MOFs/pymatgen_MOFs/' #folder to save only unique CIFs

mofs = [] #initialize list to store Pymatgen structures
entries = os.listdir(folder) #get all CIFs
entries.sort() #alphabetical sort

#for every CIF, store Pymatgen Structure in list

count = 0
for entry in entries:
    
    count = count + 1
    print(count)
    print(entry)

    if '.cif' not in entry:
        continue
    
    #read CIF
    mof_temp = Structure.from_file(os.path.join(folder,entry),primitive=False)

    #tag Pymatgen structure with its name
    mof_temp.name = entry
    mofs.append(mof_temp)

#Initialize StructureMatcher
sm = structure_matcher.StructureMatcher(primitive_cell=True)

#Group structures
groups = sm.group_structures(mofs)
print(str(len(groups))+' unique out of '+str(len(entries))+' total')

#Write out set of only unique CIFs
if not os.path.exists(new_folder):
    os.mkdir(new_folder)
for group in groups:
    mof_temp = group[0]
    mof_temp.to(filename=os.path.join(new_folder,mof_temp.name))

1
ATOXAJ_111.cif
2
ATOXAJ_121.cif
3
ATOXAJ_222.cif
4
ATOXAJ_D111.cif
5
ATOXEN_111.cif
6
ATOXEN_clean_121.cif
7
ATOXEN_clean_222.cif
8
ATOXEN_clean_D111.cif
9
BEYSEF_111.cif
10
BEYSEF_clean_222.cif
11
OLEKUM_111.cif
12
OLEKUM_D111.cif


/Users/shubhamjamdade/opt/anaconda3/lib/python3.9/site-packages/pymatgen/io/cif.py:1160: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: %s" % "\n".join(self.warnings))


13
OLEKUM_M111.cif
14
OLEKUM_manual_121.cif
15
OLEKUM_manual_222.cif
16
PARPII_111.cif
17
PARPII_121.cif
18
PARPII_222.cif
19
PARPII_D111.cif
20
QAVWAN_111.cif
21
QAVWAN_121.cif
22
QAVWAN_D111.cif
23
QAVWAN_NO_CH3_111.cif
24
QAVWAN_NO_CH3_121.cif
25
QAVWAN_NO_CH3_222.cif
26
QAVWAN_NO_CH3_D111.cif
27
QAVWAN_clean_222.cif
28
QAWVOR_111.cif
29
QOWRAV02_clean_121.cif
30
QOWRAV02_clean_222.cif
31
QOWRAV_D111.cif
32
WIMBIG_111.cif
33
WIMBIG_121.cif
34
WIMBIG_222.cif
35
WIMBIG_D111.cif
36
WONYEG_111.cif
37
WONYEG_121.cif
38
WONYEG_222.cif
39
WONYEG_D111.cif
40
ZONBAH_111.cif
41
ZONBAH_D111.cif
42
ZONBAH_clean_121.cif
43
ZONBAH_clean_222.cif
44
ZORFAQ_111.cif
45
ZORFAQ_121.cif
46
ZORFAQ_222.cif
47
ZORFAQ_D111.cif
46 unique out of 47 total


In [14]:
#This script converts a folder of CIFs to an ASE-formatted appended XYZ file and refcodes .csv file.

from ase.io import read, write
import os

cif_path = r'/Users/shubhamjamdade/Desktop/MOF_ready_for_water_simulations/ALL_DEFECTS/MOFs/'
cifs = os.listdir(cif_path)
cifs.sort()

refcodes = []
mofs = []
for cif in cifs:
    refcodes.append(cif.split('.cif')[0])
    mofs.append(read(os.path.join(cif_path, cif)))
write('mofs.xyz', mofs)

with open('refcodes.csv','w') as w:
    for refcode in refcodes:
        if refcode == refcodes[-1]:
            w.write(refcode)
        else:
            w.write(refcode+',')

In [15]:
#This script converts an ASE-formatted appended XYZ file to a folder of CIFs.

from ase.io import read, write
import numpy as np
import os

# Converts an appended .xyz to a folder of CIFs

# Relevant filenames
refcode_path = r'/Users/shubhamjamdade/Desktop/MOF_ready_for_water_simulations/ALL_DEFECTS/refcodes.csv' # path to refcodes
xyz_path = r'/Users/shubhamjamdade/Desktop/MOF_ready_for_water_simulations/ALL_DEFECTS/mofs.xyz' # path to XYZ of all structures
new_folder = r'/Users/shubhamjamdade/Desktop/MOF_ready_for_water_simulations/ALL_DEFECTS/ASE_CIFs' # path to new folder store CIFs

# ----------------------
refs = np.genfromtxt(refcode_path,delimiter=',',dtype=str)
mofs = read(xyz_path,index=':')

if not os.path.exists(new_folder):
    os.mkdir(new_folder)
for i, mof in enumerate(mofs):
    write(os.path.join(new_folder,refs[i]+'.cif'),mof)

In [ ]:
# import os
# from pathlib import Path
# folder = r'/Users/shubhamjamdade/Desktop/MOF_ready_for_water_simulations/Final_NO_DFT_NO_Charges/Cleaned /Computation Ready/'

# entries = os.listdir(folder) 
# entries.sort() #alphabetical sort
# for entry in entries:
#     p = Path(str(folder)+str(entry))
#     p.rename(p.with_suffix('.txt'))